In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install datasets diffusers transformers accelerate safetensors



In [ ]:
!nvidia-smi

NVIDIA-SMI has failed because it couldn't communicate with the NVIDIA driver. Make sure that the latest NVIDIA driver is installed and running.



In [ ]:
!apt-get install cuda

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cpp-12 cuda-12-8 cuda-cccl-12-8 cuda-command-line-tools-12-8
  cuda-compiler-12-8 cuda-crt-12-8 cuda-cudart-12-8 cuda-cudart-dev-12-8
  cuda-cuobjdump-12-8 cuda-cupti-12-8 cuda-cupti-dev-12-8 cuda-cuxxfilt-12-8
  cuda-demo-suite-12-8 cuda-documentation-12-8 cuda-driver-dev-12-8
  cuda-gdb-12-8 cuda-libraries-12-8 cuda-libraries-dev-12-8 cuda-nsight-12-8
  cuda-nsight-compute-12-8 cuda-nsight-systems-12-8 cuda-nvcc-12-8
  cuda-nvdisasm-12-8 cuda-nvml-dev-12-8 cuda-nvprof-12-8 cuda-nvprune-12-8
  cuda-nvrtc-12-8 cuda-nvrtc-dev-12-8 cuda-nvtx-12-8 cuda-nvvm-12-8
  cuda-nvvp-12-8 cuda-opencl-12-8 cuda-opencl-dev-12-8 cuda-profiler-api-12-8
  cuda-runtime-12-8 cuda-sanitizer-12-8 cuda-toolkit-12-8
  cuda-toolkit-12-8-config-common cuda-tools-12-8 cuda-visual-tools-12-8
  dctrl-tools default-jre default-jre-headless dkms fakeroot fonts-dejavu

In [ ]:
!pip install diffusers

In [ ]:
!nvidia-smi

Sat Apr 26 01:26:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers

In [ ]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [10]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("sahirp/deepfashion2")

# Example: Prepare the images and labels
def preprocess_data(example):
    # Your preprocessing code here, e.g., resizing or normalizing images
    return example

dataset = dataset.map(preprocess_data)

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset

In [9]:
from diffusers import DDPMScheduler, UNet2DModel

# Setup our custom diffusion model
# Define model and scheduler
model = UNet2DModel(
    sample_size=64,  # Adjust based on your dataset resolution
    in_channels=3,   # RGB images
    out_channels=3,
    layers_per_block=2
)

scheduler = DDPMScheduler(num_train_timesteps=1000)

In [11]:
from torch.utils.data import DataLoader
from transformers import AdamW
import torch

# Define data loader
train_dataloader = DataLoader(dataset["train"], batch_size=16, shuffle=True)

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(10):  # Number of epochs
    for batch in train_dataloader:
        optimizer.zero_grad()
        images = batch["image"].to(device)

        # Generate noise and noisy images
        noise = torch.randn_like(images).to(device)
        noisy_images = scheduler.add_noise(images, noise)

        # Forward pass
        predicted_noise = model(noisy_images)

        # Loss calculation
        loss = torch.nn.functional.mse_loss(predicted_noise, noise)

        # Backward pass
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} completed with loss: {loss.item()}")


ImportError: cannot import name 'AdamW' from 'transformers' (/usr/local/lib/python3.11/dist-packages/transformers/__init__.py)

In [ ]:
model.save_pretrained("./fashion_model")

# To evaluate: generate new images
sampled_images = model(torch.randn(16, 3, 64, 64).to(device))  # Example usage
